# Frenet — every-advantage configuration (Fix B+C, Fix A1+B+C)

Functionally identical to `cs269_dagshub_best_ever_frenet.ipynb` with the additional Fix B+C and Fix A1+B+C variants used to produce the paper's targeted-fix results.

**Configurations produced**
- Original best-ever Frenet (ADE 22.66m at 5k)
- Fix B+C: empirical normalization stats + non-zero centerline-gate init (ADE 21.00m at 5k — paper Table 1)
- Fix A1+B+C: additionally disables `tanh(d/3)` lateral compression (ADE 21.45m at 5k)

Includes a validation-triple diagnostic (round-trip identity, gate magnitude, normalization stats audit) and GCS auto-backup so training survives runtime interruptions.

**Setup requirements**: see `QUICKSTART.md` at the repo root before running.

# CS 269 Flow Planner — Colab Runner (DagsHub, all representations)

Shared notebook for all trajectory representations. Uses DagsHub storage for the
preprocessed 80k `.npz` dataset.

**To use:** change `RUN_KINEMATIC` and `RUN_SEED` in cell 0, then Run All.

| Person | RUN_KINEMATIC |
|--------|---------------|
| Jeffrey | `'waypoints'` |
| Imaan | `'frenet'` |
| Jiali | `'va'` or `'velocity'` |

**Seeds:** 42 | 1337 | 2026 — run once per seed, keep all other settings identical.


## 0. Configuration


In [ ]:
# ── Representation to train ────────────────────────────────────────────────────
RUN_KINEMATIC        = 'frenet'   # BEST-EVER FRENET FORK   # 'waypoints' | 'frenet' | 'velocity' | 'va'
RUN_SEED             = 42            # 42 | 1337 | 2026

# ── Training hyperparameters (do not change without team discussion) ───────────
TOTAL_SCENARIOS      = 10000   # 5k train + 5k val drawn from DagsHub 80k
TRAIN_SCENARIOS      = 5000
VAL_SCENARIOS        = 5000
TRAIN_EPOCHS         = 50
TRAIN_BATCH_SIZE     = 32

WARM_UP_EPOCHS       = min(5, max(0, TRAIN_EPOCHS - 1))
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE

# Auto-derive norm stats from kinematic (do not edit)
_NORM_MAP = {
    'waypoints': 'waypoints_norm_stats',
    'frenet':    'frenet_norm_stats_v1',   # BEST-EVER: v1 matched to tanh range
    'velocity':  'va_norm_stats',
    'va':        'va_norm_stats',
}
NORM_STATS = _NORM_MAP.get(RUN_KINEMATIC, 'waypoints_norm_stats')

RUN_NAME = f'{RUN_KINEMATIC}_seed{RUN_SEED}'

# Held-out test set (best-effort — skipped if raw nuPlan data not available)
HELDOUT_SCENARIOS    = 300
HELDOUT_SKIP         = 1500

# DagsHub repo (contains both the repo and the preprocessed 80k data)
DAGSHUB_REPO = 'jialic/dagshub-drive'

# Local paths (Colab SSD)
LOCAL_ROOT           = '/content/work'
LOCAL_NUPLAN         = f'{LOCAL_ROOT}/nuplan'
LOCAL_MAPS           = f'{LOCAL_NUPLAN}/maps'
LOCAL_LOGS           = f'{LOCAL_NUPLAN}/data/cache/mini'
LOCAL_EXP            = f'{LOCAL_NUPLAN}/exp'
LOCAL_CACHE          = f'{LOCAL_ROOT}/preprocessed_cache'
LOCAL_HELDOUT_CACHE  = f'{LOCAL_ROOT}/heldout_cache'
LOCAL_RUNS           = f'{LOCAL_ROOT}/runs'
LOCAL_TB             = f'{LOCAL_ROOT}/tensorboard'
LOCAL_CKPT           = f'{LOCAL_ROOT}/checkpoints'
LOCAL_RESULTS        = f'{LOCAL_ROOT}/results'

# Code / env paths
REPO_DIR             = '/content/Flow-Planner-main-2'
FP_DIR               = f'{REPO_DIR}/flow_planner'  # our fork nests flow_planner/ inside the repo
NUPLAN_DEVKIT_DIR    = '/content/nuplan-devkit'
VENV                 = '/content/venv39'
PYTHON               = f'{VENV}/bin/python'
PIP                  = f'{VENV}/bin/pip'
CONSTRAINTS          = '/tmp/pip_constraints.txt'

HELDOUT_AVAILABLE = False

print(f'Run:            {RUN_NAME}')
print(f'Kinematic:      {RUN_KINEMATIC}')
print(f'Norm stats:     {NORM_STATS}')
print(f'Train/Val:      {TRAIN_SCENARIOS} / {VAL_SCENARIOS} scenarios')
print(f'Epochs:         {TRAIN_EPOCHS} (warmup {WARM_UP_EPOCHS}), batch: {EFFECTIVE_BATCH_SIZE}')
print(f'DagsHub repo:   {DAGSHUB_REPO}')


# ===== BEST-EVER FRENET ENV VARS =====
import os
if RUN_KINEMATIC == 'frenet':
    os.environ['FRENET_SMART_CENTERLINE'] = '1'    # Option A route-aware centerline
    os.environ['FRENET_TANH_D']           = '1'    # tanh(d/3) lateral clipping
    os.environ['FRENET_TANH_D_SCALE']     = '3.0'  # tanh scale factor
    print('[best-ever frenet] env vars set:')
    print('  FRENET_SMART_CENTERLINE=1')
    print('  FRENET_TANH_D=1, FRENET_TANH_D_SCALE=3.0')
    print(f'  NORM_STATS = {NORM_STATS}')


In [ ]:
import pathlib, subprocess
if not pathlib.Path(VENV).exists():
    print('Installing python3.9...')
    !sudo apt-get update -qq
    !sudo apt-get install -y python3.9 python3.9-venv python3.9-dev > /dev/null
    !python3.9 -m venv {VENV}
    !{PYTHON} -m pip install --upgrade pip setuptools wheel -q
else:
    print(f'venv exists at {VENV}')
print(subprocess.run([PYTHON, '--version'], capture_output=True, text=True).stdout.strip())



## 1. DagsHub login + inventory


In [ ]:

# ===== BEST-EVER FORK PATCH =====
# Clone our modified fork (with audit patches + Option A centerline + v1 norm stats)
# instead of using Jiali's DagsHub Flow-Planner-main-2 mirror.
if not pathlib.Path(REPO_DIR).exists():
    print('Cloning wimaan3/cs269-flow-planner (our modified fork)...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/wimaan3/cs269-flow-planner.git',
                    REPO_DIR], check=True)
    # The repo's flow_planner code lives at REPO_DIR/flow_planner so make a symlink
    # to match the team's expected layout (REPO_DIR/flow_planner/...)
    # Already nested in our repo — no symlink needed.
    print(f'  Cloned to {REPO_DIR}')
    # Verify our audit patches and v1 norm stats exist
    nyaml = pathlib.Path(REPO_DIR) / 'flow_planner/flow_planner/script/normalization_stats/frenet_norm_stats_v1.yaml'
    print(f'  frenet_norm_stats_v1.yaml present: {nyaml.exists()}')
else:
    print(f'  REPO_DIR exists at {REPO_DIR} — skipping clone')

# Skip the DagsHub repo copy below since we have our own clone
_SKIP_DAGSHUB_REPO_COPY = True
import pathlib, time, random, shutil, subprocess

# ── Install + login to DagsHub (opens a link, no manual config needed) ───────
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'dagshub', '-q'], check=True)
# Also into venv39 so subprocess Python can see it too
subprocess.run([PYTHON, '-m', 'pip', 'install', 'dagshub', '-q'], check=True)
import dagshub.colab, dagshub.storage
DAGSHUB_REPO = 'jialic/dagshub-drive'
dagshub.colab.login()

dagshub.init(repo_owner='jialic', repo_name='dagshub-drive', mlflow=False)

# ── Mount the DagsHub storage bucket ─────────────────────────────────────────
mount_path = dagshub.storage.mount(DAGSHUB_REPO)
print(f'DagsHub storage mounted at: {mount_path}')

# ── BEST-EVER FORK: clone our modified Flow Planner instead of Jiali's mirror ──
repo_src = pathlib.Path(mount_path) / 'Flow-Planner-main-2'
repo_dst = pathlib.Path(REPO_DIR)
if repo_dst.exists():
    print(f'{REPO_DIR} already present — skipping copy')
else:
    print(f'Copying repo {repo_src} -> {repo_dst} ...')
    t = time.time()
    shutil.copytree(str(repo_src), str(repo_dst))
    print(f'Done in {time.time()-t:.1f}s')

print(f'\nRepo contents:')
!ls {REPO_DIR}

# Post-clone sanity: FP_DIR must point at a valid Python project
import pathlib as _pl
_fpsetup = _pl.Path(FP_DIR) / 'setup.py'
_fpreqs  = _pl.Path(FP_DIR) / 'requirements.txt'
assert _fpsetup.exists(), f'FP_DIR={FP_DIR} has no setup.py - our fork nests flow_planner/ one level deeper'
assert _fpreqs.exists(), f'FP_DIR={FP_DIR} has no requirements.txt'
print(f'[clone-check] FP_DIR={FP_DIR} OK (setup.py + requirements.txt present)')


In [ ]:
# Inventory the data/ folder in DagsHub storage
import pathlib
from dagshub.storage import mount

mount_path = mount(DAGSHUB_REPO)
data_path  = pathlib.Path(mount_path) / 'data'
if data_path.exists():
    npz_count = sum(1 for _ in data_path.rglob('*.npz'))
    print(f'data/ folder found: {npz_count:,} .npz files')
else:
    print(f'WARNING: data/ folder not found at {data_path}')


## 2. Clone nuplan-devkit (public)


In [ ]:
# nuplan-devkit (public, no auth) — repo already copied from DagsHub in cell 1
import pathlib

NDK_PATH = pathlib.Path(NUPLAN_DEVKIT_DIR)
if NDK_PATH.exists() and (NDK_PATH / '.git').exists():
    print(f'{NUPLAN_DEVKIT_DIR} exists — skip')
else:
    if NDK_PATH.exists():
        !rm -rf {NUPLAN_DEVKIT_DIR}
    !git clone --depth 1 https://github.com/motional/nuplan-devkit.git {NUPLAN_DEVKIT_DIR}

print('\nnuplan-devkit ready')


## 3. Python 3.9 venv


In [ ]:
import pathlib, subprocess
if not pathlib.Path(VENV).exists():
    print('Installing python3.9...')
    !sudo apt-get update -qq
    !sudo apt-get install -y python3.9 python3.9-venv python3.9-dev > /dev/null
    !python3.9 -m venv {VENV}
    !{PYTHON} -m pip install --upgrade pip setuptools wheel -q
else:
    print(f'venv exists at {VENV}')
print(subprocess.run([PYTHON, '--version'], capture_output=True, text=True).stdout.strip())


## 4. Install dependencies


In [ ]:
import pathlib
CACHED_REQS = '/content/work/working_requirements.txt'
FAST = pathlib.Path(CACHED_REQS).exists()
print(f'Install path: {"FAST" if FAST else "SLOW"}')
if FAST:
    print(f'  Using cached requirements from {CACHED_REQS}')
else:
    print('  Will install from scratch (~15-20 min)')

# Constraints file pinning numpy<2 (must persist across all pip invocations)
with open(CONSTRAINTS, 'w') as f:
    f.write('numpy<2\n')


In [ ]:
# FAST path: install from captured working_requirements.txt with --no-deps.
# SLOW path: install nuplan-devkit requirements with filtering for broken pins.
if FAST:
    import pathlib
    clean = pathlib.Path('/tmp/working_clean_reqs.txt')
    with open(CACHED_REQS) as fin, clean.open('w') as fout:
        for line in fin:
            s = line.strip()
            if (not s or s.startswith('#') or s.startswith('-e ') or '@ file://' in s
                or 'flow_planner' in s or 'nuplan-devkit' in s or 'diffusion_planner' in s):
                continue
            fout.write(line)
    !{PIP} install -r {clean} --no-deps 2>&1 | tail -10
else:
    req_in = f'{NUPLAN_DEVKIT_DIR}/requirements.txt'
    req_out = '/tmp/filtered_requirements.txt'
    bad_pins = ['hydra-core', 'omegaconf']
    with open(req_in) as f, open(req_out, 'w') as g:
        for line in f:
            if not any(line.lower().startswith(b) for b in bad_pins):
                g.write(line)
    !{PIP} install -r {req_out} -c {CONSTRAINTS} 2>&1 | tail -15
    !{PIP} install 'hydra-core>=1.2,<1.4' 'omegaconf>=2.2,<2.4' -c {CONSTRAINTS}
    !{PIP} install -r {FP_DIR}/requirements.txt -c {CONSTRAINTS} 2>&1 | tail -10


In [ ]:
# Editable install of nuplan-devkit + flow_planner so our modifications take effect
!{PIP} install -e {NUPLAN_DEVKIT_DIR} --no-deps 2>&1 | tail -3
!{PIP} install -e {FP_DIR} --no-deps 2>&1 | tail -3


In [ ]:
# Defense-in-depth: pin numpy<2
!{PIP} install 'numpy<2' --force-reinstall --no-deps -q
!{PYTHON} -c 'import numpy; print("numpy", numpy.__version__)'


## 5. Sanity check (Python 3.9-safe)


In [ ]:
# ===== DEPENDENCY SAFETY NET =====
# Verifies all critical packages are importable in the venv AFTER the SLOW path runs.
# If anything is missing (rare, but happens on cold Vertex AI L4 runtimes), force-installs.
# Idempotent: no-op when the previous cells already installed everything.
import subprocess, sys

CRITICAL_IMPORTS = [
    ('torch',                 'torch==2.3.0'),
    ('torchvision',           'torchvision==0.18.0'),
    ('flow_matching',         'flow-matching'),
    ('einops',                'einops==0.8.0'),
    ('timm',                  'timm==1.0.10'),
    ('hydra',                 'hydra-core==1.3.2'),
    ('omegaconf',             'omegaconf==2.3.0'),
    ('tensorboard',           'tensorboard==2.11.2'),
    ('pytorch_lightning',     'pytorch-lightning'),
    ('transformers',          'transformers'),
    ('aioboto3',              'aioboto3'),
    ('pyquaternion',          'pyquaternion'),
    ('matplotlib',            'matplotlib'),
    ('PIL',                   'pillow'),
    ('tqdm',                  'tqdm'),
    ('yaml',                  'pyyaml'),
    ('scipy',                 'scipy==1.13.1'),
    ('nuplan',                None),    # editable, no pip fallback
    ('flow_planner',          None),    # editable, no pip fallback
]

def check(module):
    r = subprocess.run([PYTHON, '-c', f'import {module}'], capture_output=True, text=True)
    return r.returncode == 0

missing_pip = []
missing_editable = []
for module, pkg in CRITICAL_IMPORTS:
    if not check(module):
        if pkg is None:
            missing_editable.append(module)
        else:
            missing_pip.append(pkg)

if missing_pip:
    print(f'[safety-net] missing pip packages: {missing_pip}')
    subprocess.run(
        [PIP, 'install'] + missing_pip + ['-c', CONSTRAINTS],
        check=True,
    )
if missing_editable:
    print(f'[safety-net] missing editable installs: {missing_editable}')
    if 'nuplan' in missing_editable:
        subprocess.run([PIP, 'install', '-e', NUPLAN_DEVKIT_DIR, '--no-deps'], check=True)
    if 'flow_planner' in missing_editable:
        subprocess.run([PIP, 'install', '-e', FP_DIR, '--no-deps'], check=True)

# Final re-check
still_missing = [m for m, _ in CRITICAL_IMPORTS if not check(m)]
if still_missing:
    raise RuntimeError(f'[safety-net] STILL missing after install: {still_missing}')
print('[safety-net] all critical imports OK')


In [ ]:
import pathlib, subprocess

sanity_script = '''
import sys
sys.path.insert(0, "''' + FP_DIR + '''")
import torch
import nuplan
import flow_planner
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"torch        {torch.__version__}")
print(f"cuda avail   {torch.cuda.is_available()}")
print(f"cuda device  {device_name}")
print(f"nuplan       {nuplan.__file__}")
print(f"flow_planner {flow_planner.__file__}")
print("import OK")
'''
pathlib.Path('/tmp/sanity.py').write_text(sanity_script)
r = subprocess.run([PYTHON, '/tmp/sanity.py'], capture_output=True, text=True)
print(r.stdout)
all_ok = r.returncode == 0
if not all_ok:
    print('STDERR:'); print(r.stderr)
print(f'\nall_ok = {all_ok}')
assert all_ok, 'Sanity check failed — fix above before continuing'


## 6. Capture working_requirements.txt for future FAST installs


In [ ]:
import pathlib
CACHED_REQS = '/content/work/working_requirements.txt'
if all_ok and not FAST:
    pathlib.Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
    print(f'Capturing {CACHED_REQS}...')
    !{PIP} freeze > {CACHED_REQS}
    print('Done')
else:
    print('Skipping (FAST path already used existing file)')


## 7. Get training data from DagsHub storage (`data/` folder)

Samples `.npz` files from the mounted DagsHub storage bucket (`data/` subfolder),
copies them to local SSD, then creates a deterministic 5k/5k train/val split.


In [ ]:
# dagshub already installed in cell 1 — nothing to do here
print('dagshub ready')


In [ ]:
import pathlib, time, random, shutil
import dagshub.storage

DAGSHUB_REPO = 'jialic/dagshub-drive'
mount_path   = dagshub.storage.mount(DAGSHUB_REPO)

# 80k .npz files live in the 'data' subfolder of the DagsHub storage bucket
source_cache = pathlib.Path(mount_path) / 'data'
local_cache  = pathlib.Path(LOCAL_CACHE)
local_cache.mkdir(parents=True, exist_ok=True)

all_source = sorted(source_cache.rglob('*.npz'))
print(f'80k source: {len(all_source):,} .npz files available')
print(f'Sampling  : {TOTAL_SCENARIOS} ({TRAIN_SCENARIOS} train + {VAL_SCENARIOS} val)')

rng = random.Random(RUN_SEED)
sampled = list(all_source)
rng.shuffle(sampled)
sampled = sampled[:TOTAL_SCENARIOS]

already = {p.name for p in local_cache.glob('*.npz')}
to_copy = [p for p in sampled if p.name not in already]
print(f'Already in local cache: {len(already)}  |  To copy: {len(to_copy)}')

t = time.time()
for p in to_copy:
    shutil.copy2(p, local_cache / p.name)
print(f'Done in {time.time()-t:.1f}s  |  Local cache: {len(list(local_cache.glob("*.npz")))} files')


In [ ]:
import pathlib, numpy as np

bad = []
for p in pathlib.Path(LOCAL_CACHE).glob('*.npz'):
    try:
        np.load(p)
    except Exception:
        bad.append(p)

print(f'Found {len(bad)} corrupted files — removing')
for p in bad:
    p.unlink()
print('Done — rerun Cell 7 to re-copy them')

In [ ]:
import json, pathlib, random

npz_files = sorted(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
print(f'{len(npz_files)} .npz files in {LOCAL_CACHE}')
assert len(npz_files) >= TOTAL_SCENARIOS, f'Too few .npz files ({len(npz_files)}) — copy may have failed'

# ── Deterministic 5k/5k split (seeded so it is reproducible) ─────────────────
rng = random.Random(RUN_SEED)
all_names = [p.name for p in npz_files[:TOTAL_SCENARIOS]]
shuffled  = all_names[:]
rng.shuffle(shuffled)

val_names   = set(shuffled[:VAL_SCENARIOS])
train_names = [n for n in shuffled if n not in val_names][:TRAIN_SCENARIOS]

train_json = pathlib.Path(LOCAL_CACHE) / 'train_split.json'
val_json   = pathlib.Path(LOCAL_CACHE) / 'val_split.json'
train_json.write_text(json.dumps(train_names))
val_json.write_text(json.dumps(list(val_names)))

print(f'Train split : {len(train_names)} scenarios  -> {train_json}')
print(f'Val   split : {len(val_names)}  scenarios  -> {val_json}')
assert not (set(train_names) & val_names), 'OVERLAP DETECTED — abort'
print('No train/val overlap confirmed.')

TRAIN_JSON = str(train_json)
VAL_JSON   = str(val_json)


### 7b. FALLBACK (only if cell 7 returned 0 files)

Run the preprocessor against locally available nuPlan data. Skip if cell 7 worked.


In [ ]:
# 7b FALLBACK — only runs if DagsHub cell 7 returned 0 .npz files.
# Requires raw nuPlan zips at LOCAL_ROOT/_zips/*.zip. Skips cleanly otherwise.
import pathlib, zipfile

_cache_has_npz = pathlib.Path(LOCAL_CACHE).exists() and any(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
if _cache_has_npz:
    print(f'[7b] LOCAL_CACHE already has .npz files — skipping fallback')
else:
    pathlib.Path(LOCAL_MAPS).mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_LOGS).mkdir(parents=True, exist_ok=True)
    local_zip_dir = f'{LOCAL_ROOT}/_zips'
    pathlib.Path(local_zip_dir).mkdir(parents=True, exist_ok=True)
    zips = sorted(pathlib.Path(local_zip_dir).glob('*.zip'))
    if not zips:
        print(f'[7b] No zips at {local_zip_dir} — fallback skipped (DagsHub cell 7 must succeed)')
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                target = LOCAL_MAPS if 'map' in z.name.lower() else LOCAL_LOGS
                print(f'Unzipping {z.name} -> {target} ..')
                import time; t = time.time()
                zf.extractall(target)
                print(f'  done in {time.time()-t:.1f}s')
        except zipfile.BadZipFile:
            print(f'  ERROR: {z} is corrupted.')


In [ ]:
# 7b post-preprocess — only runs if raw nuPlan data is present AND LOCAL_CACHE is empty.
import os, json, pathlib

_cache_has_npz = pathlib.Path(LOCAL_CACHE).exists() and any(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
_have_raw = (pathlib.Path(LOCAL_LOGS).exists() and any(pathlib.Path(LOCAL_LOGS).iterdir())
             and pathlib.Path(LOCAL_MAPS).exists() and any(pathlib.Path(LOCAL_MAPS).iterdir()))

if _cache_has_npz:
    print(f'[7b-preprocess] LOCAL_CACHE has .npz files — skipping preprocess')
elif not _have_raw:
    print('[7b-preprocess] No raw nuPlan data — skipping preprocess (DagsHub cell 7 must succeed)')
else:
    os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
    os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
    os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP
    %cd {FP_DIR}
    !{PYTHON} -m flow_planner.run_script.preprocess \
        --data_path {LOCAL_LOGS} \
        --map_path {LOCAL_MAPS} \
        --save_path {LOCAL_CACHE} \
        --total_scenarios {TOTAL_SCENARIOS}
    npz_files = sorted(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
    print(f'{len(npz_files)} .npz files')
    (pathlib.Path(LOCAL_CACHE) / 'diffusion_planner_training.json').write_text(
        json.dumps([p.name for p in npz_files])
    )


## 8. Held-out test set (best-effort — skips cleanly if raw data unavailable)

Tries to preprocess 300 fresh scenarios (offset 1500). Needs unzipped nuPlan data,
which we may not have if cell 7 used the cache shortcut. If anything fails here,
`HELDOUT_AVAILABLE` stays `False` and held-out eval in section 11 is skipped.


In [ ]:
# Try to set up held-out data from locally unzipped nuPlan
import pathlib

HELDOUT_AVAILABLE = False

logs_dir = pathlib.Path(LOCAL_LOGS)
maps_dir = pathlib.Path(LOCAL_MAPS)
have_raw = (logs_dir.exists() and any(logs_dir.iterdir())
            and maps_dir.exists() and any(maps_dir.iterdir()))

if have_raw:
    HELDOUT_AVAILABLE = True
    print('Raw nuPlan data present locally — proceeding with held-out preprocess')
else:
    print('Raw nuPlan data not found — held-out eval will be skipped.')
    print('(Run cell 7b fallback first if you have nuPlan zips.)')


In [ ]:
# If raw data is OK, preprocess held-out scenarios (300 starting at offset 1500)
import os, json, pathlib

if HELDOUT_AVAILABLE:
    os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
    os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
    os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP

    %cd {FP_DIR}
    !{PYTHON} -m flow_planner.run_script.preprocess \
        --data_path {LOCAL_LOGS} \
        --map_path {LOCAL_MAPS} \
        --save_path {LOCAL_HELDOUT_CACHE} \
        --total_scenarios {HELDOUT_SCENARIOS} \
        --skip_scenarios {HELDOUT_SKIP}

    heldout_files = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
    print(f'\n{len(heldout_files)} held-out .npz files')

    if len(heldout_files) == 0:
        print('Preprocess returned 0 files — held-out skipped')
        HELDOUT_AVAILABLE = False
    else:
        # Sanity: no overlap with training set
        train_set = set(p.name for p in pathlib.Path(LOCAL_CACHE).glob('*.npz'))
        overlap = train_set & set(p.name for p in heldout_files)
        if overlap:
            print(f'WARNING: {len(overlap)} train/heldout overlap, picking first non-overlapping')
            heldout_files = [p for p in heldout_files if p.name not in train_set]
        (pathlib.Path(LOCAL_HELDOUT_CACHE) / 'diffusion_planner_training.json').write_text(
            json.dumps([p.name for p in heldout_files])
        )
        print(f'Held-out ready: {len(heldout_files)} unique scenarios')
else:
    print('HELDOUT_AVAILABLE=False — skipping')

## 9. Unit tests


In [ ]:
# Unit tests — best-effort, non-fatal. Failures here don't kill training.
import pathlib, subprocess
test_dir = pathlib.Path(f'{REPO_DIR}/tests')
if not test_dir.exists():
    print('No tests/ directory — skipping')
else:
    print(f'Running tests at {test_dir} (best-effort)..')
    r = subprocess.run(
        [PYTHON, '-m', 'pytest', str(test_dir), '-v', '--tb=line', '-x', '--no-header'],
        capture_output=True, text=True, timeout=120,
    )
    # Show last 30 lines regardless
    out = (r.stdout + r.stderr).splitlines()
    for line in out[-30:]:
        print(line)
    if r.returncode != 0:
        print(f'[tests] returncode={r.returncode} — IGNORING (best-effort), continuing pipeline')


---
## 10. Experiment — train & evaluate `{RUN_KINEMATIC}`

All cells below are representation-agnostic. Change `RUN_KINEMATIC` in cell 0
to retarget a different representation without editing anything here.


### 10.1 Back up any existing checkpoint for this run name


In [ ]:
import pathlib

RUN_NAME = f'{RUN_KINEMATIC}_seed{RUN_SEED}'
pathlib.Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

existing = pathlib.Path(f'{LOCAL_CKPT}/{RUN_NAME}.ckpt')
backup   = pathlib.Path(f'{LOCAL_CKPT}/{RUN_NAME}_prev.ckpt')
if existing.exists() and not backup.exists():
    import shutil
    shutil.copy(existing, backup)
    print(f'Backed up {existing.name} -> {backup.name}')
elif backup.exists():
    print(f'{backup.name} already exists — skipping backup')
else:
    print(f'No existing {existing.name} to back up (fresh run)')


### 10.2 Measure normalization stats from train split

Computes mean/std of the model's output coordinates from training scenarios only.
The script is kinematic-aware: frenet uses (s, d), all others use (x, y).


In [ ]:
import subprocess, pathlib, re, json

stats_script = f'''
import sys; sys.path.insert(0, "{FP_DIR}")
import torch, numpy as np, pathlib, json

KINEMATIC = "{RUN_KINEMATIC}"
train_names = json.loads(pathlib.Path("{LOCAL_CACHE}/train_split.json").read_text())
npz_files   = [pathlib.Path("{LOCAL_CACHE}") / n for n in train_names]
print(f"Computing stats from {{len(npz_files)}} TRAIN-ONLY scenarios (kinematic={{KINEMATIC}})...")

all_c0, all_c1 = [], []

if KINEMATIC == "frenet":
    from flow_planner.data.normalization.frenet_utils import (
        select_reference_centerline, cartesian_to_frenet
    )
    for p in npz_files[:5000]:
        data = np.load(p)
        ego_future = torch.from_numpy(data["ego_agent_future"]).to(torch.float32).unsqueeze(0)
        lanes  = torch.from_numpy(data["lanes"]).to(torch.float32).unsqueeze(0)
        routes = torch.from_numpy(data["route_lanes"]).to(torch.float32).unsqueeze(0)
        centerline = select_reference_centerline(routes, lanes)
        sd = cartesian_to_frenet(ego_future[..., :2], centerline)
        all_c0.append(sd[0, :, 0].numpy())  # s
        all_c1.append(sd[0, :, 1].numpy())  # d
    dim0_name, dim1_name = "s", "d"
elif KINEMATIC in ("va", "velocity"):
    for p in npz_files[:500]:
        data = np.load(p)
        xy = data["ego_agent_future"][:, :2]
        vx = np.diff(xy[:, 0], prepend=xy[0, 0])
        vy = np.diff(xy[:, 1], prepend=xy[0, 1])
        all_c0.append(vx)
        all_c1.append(vy)
    dim0_name, dim1_name = "vx", "vy"
else:  # waypoints — absolute (x, y)
    for p in npz_files[:500]:
        data = np.load(p)
        traj = data["ego_agent_future"]
        all_c0.append(traj[:, 0])
        all_c1.append(traj[:, 1])
    dim0_name, dim1_name = "x", "y"

c0 = np.concatenate(all_c0); c1 = np.concatenate(all_c1)
c0_mean, c0_std = round(float(c0.mean())), max(1, round(float(c0.std())))
c1_mean, c1_std = round(float(c1.mean())), max(1, round(float(c1.std())))

print(f"\n{{dim0_name}}: mean={{c0.mean():.2f}}, std={{c0.std():.2f}}, min={{c0.min():.2f}}, max={{c0.max():.2f}}")
print(f"{{dim1_name}}: mean={{c1.mean():.2f}}, std={{c1.std():.2f}}, min={{c1.min():.2f}}, max={{c1.max():.2f}}")
print(f"NEW_MEAN={{[c0_mean, c1_mean, 1, 0]}}")
print(f"NEW_STD={{[c0_std, c1_std, 0.3, 0.3]}}")
'''
pathlib.Path('/tmp/measure_stats.py').write_text(stats_script)
result = subprocess.run([PYTHON, '/tmp/measure_stats.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:'); print(result.stderr)
    raise RuntimeError('Stats measurement failed')

m_mean = re.search(r'NEW_MEAN=(\[[^\]]+\])', result.stdout)
m_std  = re.search(r'NEW_STD=(\[[^\]]+\])',  result.stdout)
assert m_mean and m_std, 'Failed to parse stats output'
NEW_MEAN = m_mean.group(1)
NEW_STD  = m_std.group(1)
print(f'\nWill write to YAML:\n  mean: {NEW_MEAN}\n  std:  {NEW_STD}')


### 10.3 Update `{NORM_STATS}.yaml` with measured values


In [ ]:
import pathlib, shutil

# Create norm stats yaml for this kinematic if it doesn't exist yet
norm_yaml = f'{FP_DIR}/flow_planner/script/normalization_stats/{NORM_STATS}.yaml'
src_yaml  = f'{FP_DIR}/flow_planner/script/normalization_stats/waypoints_norm_stats.yaml'

if not pathlib.Path(norm_yaml).exists():
    shutil.copy(src_yaml, norm_yaml)
    print(f'Created {norm_yaml} (copied from waypoints_norm_stats as template)')
else:
    print(f'Already exists: {norm_yaml}')


In [ ]:
import re, pathlib

yaml_path = f'{FP_DIR}/flow_planner/script/normalization_stats/{NORM_STATS}.yaml'
assert pathlib.Path(yaml_path).exists(), (
    f'YAML not found: {yaml_path}\n'
    f'Either create {NORM_STATS}.yaml or change NORM_STATS in cell 0.'
)
content = pathlib.Path(yaml_path).read_text()

def replace_uniform(m):
    indent = m.group(1)
    inner  = indent + '  '
    return f'{indent}uniform:\n{inner}mean: {NEW_MEAN}\n{inner}std: {NEW_STD}'

pattern     = r'(\s*)uniform:\s*\n\s*mean:\s*\[[^\]]+\]\s*\n\s*std:\s*\[[^\]]+\]'
content_new = re.sub(pattern, replace_uniform, content)
pathlib.Path(yaml_path).write_text(content_new)
print(pathlib.Path(yaml_path).read_text())


### 10.4 Train (~10 min on L4)


In [ ]:
import os, pathlib, json

os.environ['PROJECT_ROOT']         = FP_DIR
os.environ['SAVE_DIR']             = LOCAL_RUNS
os.environ['TENSORBOARD_LOG_PATH'] = LOCAL_TB
os.environ['TRAINING_DATA']        = LOCAL_CACHE
os.environ['TRAINING_JSON']        = TRAIN_JSON   # train split only — no val leakage
os.environ['WORLD_SIZE']           = '1'
os.environ['LOCAL_RANK']           = '0'
os.environ['MASTER_ADDR']          = 'localhost'
os.environ['MASTER_PORT']          = '29529'
os.environ['HYDRA_FULL_ERROR']     = '1'

pathlib.Path(LOCAL_RUNS).mkdir(parents=True, exist_ok=True)
pathlib.Path(LOCAL_TB).mkdir(parents=True, exist_ok=True)
!rm -rf {LOCAL_RUNS}/*

# Frenet needs the CenterlineEncoder wired into the encoder
FRENET_OVERRIDES = (
    '+model.model_encoder.centerline_encoder._target_=flow_planner.model.modules.encoder_modules.CenterlineEncoder '
    '+model.model_encoder.centerline_encoder.n_points=100 '
    '+model.model_encoder.centerline_encoder.hidden_dim=256'
) if RUN_KINEMATIC == 'frenet' else ''

print(f'Training {RUN_NAME} | kinematic={RUN_KINEMATIC} | norm_stats={NORM_STATS}')
print(f'Scenarios: {len(json.loads(open(TRAIN_JSON).read()))} (train split)')
if FRENET_OVERRIDES:
    print(f'Frenet overrides: {FRENET_OVERRIDES}')


In [ ]:
# ===== GCS auto-backup (survives runtime death) =====
# Syncs /content/work/runs (training) AND /content/work top-level files
# (eval JSONs, BEV PNGs, results.csv) every 120s. Skips preprocessed_cache/
# and heldout_cache/ (multi-GB, can re-fetch from DagsHub).
#
# Recover after runtime death:
#   gsutil -m cp -r gs://cs269-scale-ablation-archive/best_ever_dagshub_seed42/ ./
import os, subprocess, time, pathlib

GCS_BUCKET   = 'gs://cs269-scale-ablation-archive'
GCS_RUN_DIR  = f'{GCS_BUCKET}/best_ever_dagshub_seed{RUN_SEED}'
LOCAL_RUNS_DIR = '/content/work/runs'
LOCAL_WORK_DIR = '/content/work'
pathlib.Path(LOCAL_RUNS_DIR).mkdir(parents=True, exist_ok=True)

print('[gcs-backup] target:', GCS_RUN_DIR)
subprocess.run(['gsutil', 'ls', GCS_BUCKET], capture_output=True)

# Kill any previous syncer
subprocess.run('pkill -f gcs_backup_loop || true', shell=True)

LOOP_SH = '/content/gcs_backup_loop.sh'
with open(LOOP_SH, 'w') as f:
    f.write(f"""#!/bin/bash
# Two-tier backup: training run dir + top-level output files (no caches).
while true; do
  # Tier 1: full training run directory
  gsutil -m rsync -r {LOCAL_RUNS_DIR} {GCS_RUN_DIR}/runs > /content/gcs_backup_runs.log 2>&1
  # Tier 2: loose output files in /content/work (eval_*.json, bev_*.png, results.csv, etc.)
  for pattern in eval_*.json bev_*.png results.csv frenet_validation_*.json train.log; do
    for f in {LOCAL_WORK_DIR}/$pattern; do
      [ -f "$f" ] && gsutil cp "$f" {GCS_RUN_DIR}/$(basename "$f") > /dev/null 2>&1
    done
  done
  sleep 120
done
""")
subprocess.run(['chmod', '+x', LOOP_SH])

# Launch detached
subprocess.Popen(
    ['nohup', 'bash', LOOP_SH],
    stdout=open('/content/gcs_backup_nohup.log', 'w'),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setpgrp,
)
time.sleep(2)
print('[gcs-backup] background loop started (rsync runs/ + cp eval JSONs / BEVs / CSV / validation / train.log)')
subprocess.run(['pgrep', '-af', 'gcs_backup_loop'])


In [ ]:
# ===== BEST-EVER FRENET ENV PATCH (training) =====
# Output is redirected to /content/work/train.log to keep the notebook tab light.
# A poller prints ONLY epoch-loss lines (~1 per minute) so the cell stays small.
import os, subprocess, time, threading, pathlib

if RUN_KINEMATIC == 'frenet':
    os.environ['FRENET_SMART_CENTERLINE'] = '1'
    os.environ['FRENET_TANH_D']           = '1'
    os.environ['FRENET_TANH_D_SCALE']     = '3.0'
    print('[best-ever frenet] training env vars set')

LOG_PATH = '/content/work/train.log'
pathlib.Path('/content/work').mkdir(parents=True, exist_ok=True)
# Truncate any prior log
open(LOG_PATH, 'w').close()

TRAIN_CMD = (
    f'cd {FP_DIR} && '
    f'{VENV}/bin/torchrun --nnodes 1 --nproc-per-node 1 --standalone '
    f'  flow_planner/trainer.py --config-name flow_planner_standard '
    f'    ++model.kinematic={RUN_KINEMATIC} '
    f'    normalization_stats={NORM_STATS} '
    f'    train.batch_size={EFFECTIVE_BATCH_SIZE} '
    f'    train.epoch={TRAIN_EPOCHS} '
    f'    scheduler.warm_up_epoch={WARM_UP_EPOCHS} '
    f'    train.save_utd=1 '
    f'    ddp.distributed=false '
    f'    seed={RUN_SEED} '
    f'    job_name={RUN_NAME} '
    f'    num_workers=4 '
    f'    {FRENET_OVERRIDES} '
    f'  > {LOG_PATH} 2>&1'
)
print('[train] launching torchrun, logs ->', LOG_PATH)
proc = subprocess.Popen(TRAIN_CMD, shell=True, executable='/bin/bash', preexec_fn=os.setpgrp)
print('[train] PID:', proc.pid)

# Poll the log: print ONLY epoch_loss summary lines, not the tqdm spam
def tail_filter():
    seen = set()
    last_size = 0
    while proc.poll() is None:
        try:
            size = os.path.getsize(LOG_PATH)
            if size > last_size:
                with open(LOG_PATH, 'r', errors='ignore') as f:
                    f.seek(last_size)
                    chunk = f.read()
                last_size = size
                for line in chunk.splitlines():
                    if 'epoch loss' in line or 'epoch_lr' in line or 'Training finished' in line or 'Model saved' in line or 'ERROR' in line or 'Traceback' in line:
                        if line not in seen:
                            seen.add(line)
                            print(line, flush=True)
        except Exception:
            pass
        time.sleep(30)
    # Final flush after exit
    try:
        with open(LOG_PATH, 'r', errors='ignore') as f:
            tail = f.read()[-2000:]
        print('--- train.log tail ---')
        print(tail)
    except Exception:
        pass
    print(f'[train] process exited with code {proc.returncode}')

t = threading.Thread(target=tail_filter, daemon=False)
t.start()
t.join()


### 10.5 Save checkpoint to Drive


In [ ]:
import pathlib

ckpts = list(pathlib.Path(LOCAL_RUNS).rglob('*.ckpt')) + list(pathlib.Path(LOCAL_RUNS).rglob('*.pth'))
assert ckpts, f'No checkpoint found under {LOCAL_RUNS}'
best_ckpt = sorted(ckpts, key=lambda p: p.stat().st_mtime)[-1]
print(f'Most recent: {best_ckpt}')

pathlib.Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)
run_ckpt_path = f'{LOCAL_CKPT}/{RUN_NAME}.ckpt'
!cp {best_ckpt} {run_ckpt_path}
print(f'Saved to {run_ckpt_path}')


### 10.6 Eval on validation split (unseen during training)


In [ ]:
import json

n_val_batches = max(1, len(json.loads(open(VAL_JSON).read())) // EFFECTIVE_BATCH_SIZE)
eval_val_json = f'/content/work/eval_{RUN_KINEMATIC}_val.json'

%cd {FP_DIR}
!{PYTHON} -m flow_planner.run_script.inference_eval \
    --checkpoint {run_ckpt_path} \
    --data_dir {LOCAL_CACHE} \
    --data_list {VAL_JSON} \
    --kinematic {RUN_KINEMATIC} \
    --norm_stats {NORM_STATS} \
    --output_json {eval_val_json} \
    --batch_size {EFFECTIVE_BATCH_SIZE} \
    --num_batches {n_val_batches}

with open(eval_val_json) as f:
    run_val = json.load(f)
print(f'{RUN_KINEMATIC} on VALIDATION set (unseen during training):')
for k, v in run_val.items():
    print(f'  {k}: {v}')


# Explicit push of eval JSON to GCS (belt-and-suspenders vs background loop)
import subprocess as _sp
_gcs_target = f'gs://cs269-scale-ablation-archive/best_ever_dagshub_seed{RUN_SEED}/eval_{RUN_KINEMATIC}_val.json'
_sp.run(['gsutil', 'cp', eval_val_json, _gcs_target], check=False)
print('[push] eval JSON ->', _gcs_target)


### 10.7 BEV visualization


In [ ]:
import subprocess, pathlib

bev_out = f'/content/work/bev_{RUN_KINEMATIC}.png'

# Frenet overrides needed when loading checkpoint
_frenet_overrides_str = (
    '"+model.model_encoder.centerline_encoder._target_=flow_planner.model.modules.encoder_modules.CenterlineEncoder",'
    '"+model.model_encoder.centerline_encoder.n_points=100",'
    '"+model.model_encoder.centerline_encoder.hidden_dim=256",'
) if RUN_KINEMATIC == 'frenet' else ''

viz_script = f'''
import sys; sys.path.insert(0, "{FP_DIR}")
import torch, numpy as np, matplotlib.pyplot as plt, pathlib, json
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from flow_planner.data.dataset.nuplan import NuPlanDataset
from flow_planner.data.utils.collect import collect_batch
from torch.utils.data import DataLoader

KINEMATIC = "{RUN_KINEMATIC}"
CKPT      = "{run_ckpt_path}"
CACHE     = "{LOCAL_CACHE}"
VAL_LIST  = "{VAL_JSON}"

with initialize_config_dir(version_base=None, config_dir="{FP_DIR}/flow_planner/script"):
    cfg = compose(config_name="flow_planner_standard",
                  overrides=[f"++model.kinematic={{KINEMATIC}}",
                             f"normalization_stats={NORM_STATS}",
                             "ddp.distributed=false",
                             {_frenet_overrides_str}])

model = instantiate(cfg.model)
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, weights_only=False, map_location=device)
sd = ckpt.get("ema_state_dict", ckpt.get("state_dict", ckpt))
sd = {{k.replace("module.", ""): v for k, v in sd.items()}}
model.load_state_dict(sd, strict=False)
model = model.to(device).eval()

ds = NuPlanDataset(data_dir=CACHE, data_list=VAL_LIST,
                   past_neighbor_num=cfg.model.neighbor_num,
                   predicted_neighbor_num=cfg.model.neighbor_pred_num,
                   future_len=cfg.model.future_len,
                   future_downsampling_method="uniform")
loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=collect_batch)
batch  = next(iter(loader)).to(device)

with torch.no_grad():
    preds = model(batch, mode="inference", use_cfg=False, cfg_weight=cfg.model.cfg_weight)

if KINEMATIC == "frenet":
    from flow_planner.data.normalization.frenet_utils import frenet_to_cartesian, select_reference_centerline
    centerline = select_reference_centerline(route_lanes=batch.route_lanes, lanes=batch.lanes)
    pred_xy = frenet_to_cartesian(preds[:, 0, :, :2], centerline).cpu().numpy()
    cl_np   = centerline.cpu().numpy()
elif KINEMATIC in ("va", "velocity"):
    pred_dxy = preds[:, 0, :, :2]
    pred_xy  = pred_dxy.cumsum(dim=1).cpu().numpy()  # ego-centric: starts at (0,0)
    cl_np    = None
else:  # waypoints
    pred_xy = preds[:, 0, :, :2].cpu().numpy()
    cl_np   = None

gt_xy = batch.ego_future[:, :preds.shape[2], :2].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i in range(4):
    if cl_np is not None:
        axes[i].plot(cl_np[i, :, 0], cl_np[i, :, 1], "k--", lw=1, label="centerline")
    axes[i].plot(gt_xy[i, :, 0],   gt_xy[i, :, 1],   "g-",  lw=2, label="GT")
    axes[i].plot(pred_xy[i, :, 0], pred_xy[i, :, 1], "r--", lw=2, label="Pred")
    axes[i].scatter([0], [0], c="b", s=100, zorder=5)
    axes[i].set_title(f"Scene {{i}}")
    axes[i].axis("equal"); axes[i].legend(fontsize=8); axes[i].grid(alpha=0.3)

plt.suptitle(f"{{KINEMATIC}} seed{RUN_SEED} — val set")
plt.savefig("{bev_out}", dpi=100, bbox_inches="tight")
plt.close()
print(f"saved {bev_out}")
'''

pathlib.Path('/tmp/viz_run.py').write_text(viz_script)
r = subprocess.run([PYTHON, '/tmp/viz_run.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])

from IPython.display import Image, display
if pathlib.Path(bev_out).exists():
    display(Image(bev_out))
else:
    print(f'BEV not found: {bev_out} — check viz errors above')


In [ ]:
from IPython.display import Image, display
import pathlib
bev_out = f'/content/work/bev_{RUN_KINEMATIC}.png'
if pathlib.Path(bev_out).exists():
    display(Image(bev_out))
else:
    print(f'BEV not found: {bev_out} — check viz errors above')


### 10.6b Frenet validation triple (PROOF that Frenet is properly trained)

Three diagnostics run on the trained checkpoint + actual training cache.
All three must pass to claim "Frenet was set up correctly".

| Check | Pass criterion | What it proves |
|---|---|---|
| **A** Round-trip | max \|xy − roundtrip(xy)\| < 0.5 m | encode/decode math works on real data |
| **B** centerline_gate | gate > 0.1 | model learned to use the centerline |
| **C** (s, d) stats | empirical mean/std within 20% of YAML | normalization is correctly calibrated |


In [ ]:
# Validation A: Frenet round-trip on REAL training cache scenarios.
# Loads 100 random .npz files, runs cartesian_to_frenet then frenet_to_cartesian,
# reports max error. Math is provably correct in unit tests on synthetic data;
# this is the first time we measure it on actual nuPlan trajectories.
import sys, json, pathlib, random as _rnd
sys.path.insert(0, FP_DIR)

import numpy as np, torch
from flow_planner.data.normalization.frenet_utils import (
    select_reference_centerline, cartesian_to_frenet, frenet_to_cartesian,
)

# Match env vars from training so we hit the same code paths
import os
os.environ['FRENET_SMART_CENTERLINE'] = '1'
os.environ['FRENET_TANH_D']           = '1'
os.environ['FRENET_TANH_D_SCALE']     = '3.0'

cache_dir = pathlib.Path(LOCAL_CACHE)
all_files = list(cache_dir.glob('*.npz'))
_rnd.seed(0)
sample_files = _rnd.sample(all_files, min(100, len(all_files)))
print(f'[A] round-trip on {len(sample_files)} scenarios')

errors = []
for fp in sample_files:
    try:
        d = np.load(fp)
        ego_future = torch.from_numpy(d['ego_agent_future']).float().unsqueeze(0)   # (1, T, F)
        ego_past   = torch.from_numpy(d['ego_agent_past']).float().unsqueeze(0)
        lanes      = torch.from_numpy(d['lanes']).float().unsqueeze(0)
        routes     = torch.from_numpy(d['route_lanes']).float().unsqueeze(0)
        centerline = select_reference_centerline(
            route_lanes=routes, lanes=lanes,
            ego_past_xy=ego_past[..., :2],
        )
        xy = ego_future[..., :2]
        sd = cartesian_to_frenet(xy, centerline)
        xy_rt = frenet_to_cartesian(sd, centerline)
        err = (xy - xy_rt).abs().max().item()
        errors.append(err)
    except Exception as e:
        print(f'  skip {fp.name}: {e}')

errors = np.array(errors)
A_PASS = bool(errors.max() < 0.5)
print(f'[A] errors: median={np.median(errors):.3f}m  p90={np.percentile(errors, 90):.3f}m  max={errors.max():.3f}m')
print(f'[A] {"PASS" if A_PASS else "FAIL"}  (criterion: max < 0.5 m)')

VALIDATION_A = {'pass': A_PASS, 'max_error_m': float(errors.max()),
                'p90_error_m': float(np.percentile(errors, 90)),
                'median_error_m': float(np.median(errors)),
                'n_scenarios': len(errors)}


In [ ]:
# Validation B: read the final centerline_gate value from the trained checkpoint.
# Zero-init at start of training. If the model learned to use the centerline
# conditioning, the gate has moved away from zero. If it's still ~0, the model
# ignored Option A — likely the dominant reason for the Frenet penalty.
import torch, pathlib

ckpt_path = run_ckpt_path  # set by cell 46 (Save checkpoint)
print(f'[B] loading {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
state = ckpt.get('state_dict', ckpt.get('model', ckpt))

# Find centerline_gate parameter
gate_keys = [k for k in state.keys() if 'centerline_gate' in k]
print(f'[B] found gate keys: {gate_keys}')

if not gate_keys:
    B_PASS = False
    gate_value = None
    print('[B] FAIL  (no centerline_gate parameter found — Option A not wired in)')
else:
    gate_tensor = state[gate_keys[0]]
    gate_value = float(gate_tensor.abs().mean().item())
    B_PASS = gate_value > 0.1
    print(f'[B] |centerline_gate|.mean = {gate_value:.4f}')
    print(f'[B] {"PASS" if B_PASS else "FAIL"}  (criterion: |gate| > 0.1)')
    if not B_PASS:
        print('     ↳ gate barely moved from zero-init. Model likely ignored centerline conditioning.')
        print('       This is consistent with "Frenet without Option A" performance.')

VALIDATION_B = {'pass': B_PASS, 'gate_mean_abs': gate_value, 'param_key': gate_keys[0] if gate_keys else None}


In [ ]:
# Validation C: empirical (s, d) distribution vs frenet_norm_stats_v1.yaml.
# YAML uses mean=[31, 0, 1, 0], std=[26, 0.5, 1.0, 1.0] for the ego target.
# If empirical (s, d) on our cache differs significantly, normalization
# is off-center and flow-matching convergence suffers.
import sys, pathlib, numpy as np, torch, yaml as _yaml, random as _rnd
sys.path.insert(0, FP_DIR)
from flow_planner.data.normalization.frenet_utils import (
    select_reference_centerline, cartesian_to_frenet,
)
import os
os.environ['FRENET_SMART_CENTERLINE'] = '1'
os.environ['FRENET_TANH_D']           = '1'
os.environ['FRENET_TANH_D_SCALE']     = '3.0'

cache_dir = pathlib.Path(LOCAL_CACHE)
all_files = list(cache_dir.glob('*.npz'))
_rnd.seed(1)
sample_files = _rnd.sample(all_files, min(300, len(all_files)))
print(f'[C] measuring (s, d) over {len(sample_files)} scenarios')

s_vals, d_vals = [], []
for fp in sample_files:
    try:
        d_npz = np.load(fp)
        ego_future = torch.from_numpy(d_npz['ego_agent_future']).float().unsqueeze(0)
        ego_past   = torch.from_numpy(d_npz['ego_agent_past']).float().unsqueeze(0)
        lanes      = torch.from_numpy(d_npz['lanes']).float().unsqueeze(0)
        routes     = torch.from_numpy(d_npz['route_lanes']).float().unsqueeze(0)
        centerline = select_reference_centerline(
            route_lanes=routes, lanes=lanes, ego_past_xy=ego_past[..., :2],
        )
        sd = cartesian_to_frenet(ego_future[..., :2], centerline)
        s_vals.append(sd[0, :, 0].numpy())
        d_vals.append(sd[0, :, 1].numpy())
    except Exception:
        pass

s_arr = np.concatenate(s_vals); d_arr = np.concatenate(d_vals)
s_mean, s_std = float(s_arr.mean()), float(s_arr.std())
d_mean, d_std = float(d_arr.mean()), float(d_arr.std())

# YAML reference (Option A + tanh)
yaml_path = pathlib.Path(FP_DIR) / 'flow_planner/script/normalization_stats/frenet_norm_stats_v1.yaml'
yaml_stats = _yaml.safe_load(open(yaml_path))
y_mean = yaml_stats['ego']['uniform']['mean']
y_std  = yaml_stats['ego']['uniform']['std']
print(f'[C] YAML:        mean=[{y_mean[0]}, {y_mean[1]}, ..]  std=[{y_std[0]}, {y_std[1]}, ..]')
print(f'[C] empirical:   s mean={s_mean:.2f}  std={s_std:.2f}')
print(f'[C] empirical:   d mean={d_mean:.4f}  std={d_std:.4f}')

# Within 20%
def _within(empirical, target, tol=0.2):
    return abs(empirical - target) <= max(tol * abs(target), 0.1)

C_PASS = (_within(s_mean, y_mean[0]) and _within(s_std, y_std[0])
          and _within(d_mean, y_mean[1]) and _within(d_std, y_std[1]))
print(f'[C] {"PASS" if C_PASS else "FAIL"}  (criterion: empirical within 20% of YAML)')

VALIDATION_C = {'pass': C_PASS,
                'empirical': {'s_mean': s_mean, 's_std': s_std, 'd_mean': d_mean, 'd_std': d_std},
                'yaml':      {'s_mean': y_mean[0], 's_std': y_std[0], 'd_mean': y_mean[1], 'd_std': y_std[1]}}


In [ ]:
# Validation summary — three checks together prove Frenet is properly trained.
import json, pathlib
VALIDATION = {
    'A_round_trip':       VALIDATION_A,
    'B_centerline_gate':  VALIDATION_B,
    'C_sd_distribution':  VALIDATION_C,
    'all_pass': bool(VALIDATION_A['pass'] and VALIDATION_B['pass'] and VALIDATION_C['pass']),
}
out_path = pathlib.Path(f'/content/work/frenet_validation_{RUN_NAME}.json')
out_path.write_text(json.dumps(VALIDATION, indent=2))
print(json.dumps(VALIDATION, indent=2))
print()
print('=' * 60)
print(f'OVERALL: {"PROVEN — Frenet pipeline is correctly set up" if VALIDATION["all_pass"] else "ONE OR MORE CHECKS FAILED — see details above"}')
print('=' * 60)

# Push to GCS so we keep the validation report even if runtime dies
import subprocess
gcs_target = f'gs://cs269-scale-ablation-archive/best_ever_dagshub_seed{RUN_SEED}/frenet_validation_{RUN_NAME}.json'
subprocess.run(['gsutil', 'cp', str(out_path), gcs_target], check=False)
print(f'\nUploaded validation report to {gcs_target}')


### 10.8 Save artifacts + append row to results.csv


In [ ]:
import shutil, pathlib, datetime, csv, json

exp_id       = f'{RUN_NAME}_scen{TOTAL_SCENARIOS}_ep{TRAIN_EPOCHS}'
artifact_dir = pathlib.Path(f'{LOCAL_ROOT}/experiments/{exp_id}')
artifact_dir.mkdir(parents=True, exist_ok=True)

bev_out = f'/content/work/bev_{RUN_KINEMATIC}.png'
if pathlib.Path(bev_out).exists():
    shutil.copy(bev_out, artifact_dir / 'bev.png')
shutil.copy(eval_val_json, artifact_dir / 'eval_val.json')
yaml_path = f'{FP_DIR}/flow_planner/script/normalization_stats/{NORM_STATS}.yaml'
shutil.copy(yaml_path, artifact_dir / f'{NORM_STATS}.yaml')

notes = (
    f'Experiment: {exp_id}\n'
    f'Date: {datetime.datetime.now().isoformat(timespec="seconds")}\n\n'
    f'Config: kinematic={RUN_KINEMATIC}, norm_stats={NORM_STATS}\n'
    f'        train={TRAIN_SCENARIOS}, val={VAL_SCENARIOS}, '
    f'epochs={TRAIN_EPOCHS}, batch={EFFECTIVE_BATCH_SIZE}, seed={RUN_SEED}\n'
    f'Norm stats: mean={NEW_MEAN}, std={NEW_STD}\n\n'
    f'Val ADE: {run_val["ade_mean"]:.3f} +- {run_val["ade_std"]:.3f} m\n'
    f'Val FDE: {run_val["fde_mean"]:.3f} +- {run_val["fde_std"]:.3f} m\n'
)
(artifact_dir / 'NOTES.md').write_text(notes)
print(f'Artifacts -> {artifact_dir}')

# Append row to results.csv
pathlib.Path(LOCAL_RESULTS).mkdir(parents=True, exist_ok=True)
results_csv = f'{LOCAL_RESULTS}/results.csv'
row = {
    'run_name': RUN_NAME, 'kinematic': RUN_KINEMATIC, 'seed': RUN_SEED,
    'train_scenarios': TRAIN_SCENARIOS, 'val_scenarios': VAL_SCENARIOS,
    'epochs': TRAIN_EPOCHS,
    'batch_size': EFFECTIVE_BATCH_SIZE, 'split': 'val',
    'ade_mean': run_val['ade_mean'], 'ade_std': run_val['ade_std'],
    'fde_mean': run_val['fde_mean'], 'fde_std': run_val['fde_std'],
    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
}
exists = pathlib.Path(results_csv).exists()
with open(results_csv, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    if not exists:
        w.writeheader()
    w.writerow(row)
print(f'Appended to {results_csv}')


# Explicit push of results.csv (val) to GCS
import subprocess as _sp, pathlib as _pl
_csv_path = _pl.Path(f'{LOCAL_RESULTS}/results.csv')
if _csv_path.exists():
    _sp.run(['gsutil', 'cp', str(_csv_path), f'gs://cs269-scale-ablation-archive/best_ever_dagshub_seed{RUN_SEED}/results.csv'], check=False)
    print('[push] results.csv (val) -> GCS')


---
## 11. Held-out eval (runs only if HELDOUT_AVAILABLE)


In [ ]:
if not HELDOUT_AVAILABLE:
    print('Skipping held-out eval (raw nuPlan data not available)')
else:
    print(f'Running held-out eval on {HELDOUT_SCENARIOS} scenarios...')


In [ ]:
if HELDOUT_AVAILABLE:
    import json
    eval_heldout_json = f'/content/work/eval_{RUN_KINEMATIC}_heldout.json'
    %cd {FP_DIR}
    !{PYTHON} -m flow_planner.run_script.inference_eval \
        --checkpoint {run_ckpt_path} \
        --data_dir {LOCAL_HELDOUT_CACHE} \
        --data_list {LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json \
        --kinematic {RUN_KINEMATIC} --norm_stats {NORM_STATS} \
        --output_json {eval_heldout_json} \
        --batch_size {EFFECTIVE_BATCH_SIZE} --num_batches 100
    with open(eval_heldout_json) as f:
        run_heldout = json.load(f)
    print(f'\n{RUN_KINEMATIC} on HELD-OUT:')
    for k, v in run_heldout.items():
        print(f'  {k}: {v}')


In [ ]:
# Save held-out row to results.csv
if HELDOUT_AVAILABLE and 'run_heldout' in dir():
    import csv, datetime, pathlib
    results_csv = f'{LOCAL_RESULTS}/results.csv'
    row = {
        'run_name': RUN_NAME, 'kinematic': RUN_KINEMATIC, 'seed': RUN_SEED,
        'train_scenarios': TRAIN_SCENARIOS, 'val_scenarios': VAL_SCENARIOS,
        'epochs': TRAIN_EPOCHS,
        'batch_size': EFFECTIVE_BATCH_SIZE, 'split': 'heldout',
        'ade_mean': run_heldout['ade_mean'], 'ade_std': run_heldout['ade_std'],
        'fde_mean': run_heldout['fde_mean'], 'fde_std': run_heldout['fde_std'],
        'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
    }
    exists = pathlib.Path(results_csv).exists()
    with open(results_csv, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if not exists:
            w.writeheader()
        w.writerow(row)
    print(f'  appended {RUN_KINEMATIC} held-out')


# Explicit push of results.csv (heldout) to GCS
if HELDOUT_AVAILABLE:
    import subprocess as _sp
    _sp.run(['gsutil', 'cp', f'{LOCAL_RESULTS}/results.csv', f'gs://cs269-scale-ablation-archive/best_ever_dagshub_seed{RUN_SEED}/results.csv'], check=False)
    print('[push] results.csv (heldout) -> GCS')


---
## 12. Final comparison table


In [ ]:
import csv, pathlib
results_csv = f'{LOCAL_RESULTS}/results.csv'

with open(results_csv) as f:
    rows = list(csv.DictReader(f))

# Latest row per (kinematic, split)
latest = {}
for r in rows:
    key = (r['kinematic'], r.get('split', 'val'))
    latest[key] = r

print(f'{"Rep":<14} {"Split":<10} {"ADE (m)":<20} {"FDE (m)":<20} {"Run"}')
print('-' * 90)
for (rep, split), r in sorted(latest.items()):
    ade = f'{float(r["ade_mean"]):.2f} +- {float(r["ade_std"]):.2f}'
    fde = f'{float(r["fde_mean"]):.2f} +- {float(r["fde_std"]):.2f}'
    print(f'{rep:<14} {split:<10} {ade:<20} {fde:<20} {r["run_name"]}')


In [ ]:
# Display all available BEVs
from IPython.display import Image, display
import pathlib, glob

for bev in sorted(pathlib.Path('/content/work').glob('bev_*.png')):
    print(f'=== {bev.stem} ===')
    display(Image(str(bev)))


---
## Done

**To run a different representation:** change `RUN_KINEMATIC` in cell 0 and re-run cells 0, 10.1–10.8.
All other cells are shared infrastructure and do not need to change.


In [ ]:
# ===== Final config dump (best-ever Frenet verification) =====
import os
print('=== BEST-EVER FRENET RUN CONFIG ===')
print(f'RUN_KINEMATIC        = {RUN_KINEMATIC}')
print(f'RUN_SEED             = {RUN_SEED}')
print(f'NORM_STATS           = {NORM_STATS}')
print(f'TRAIN_SCENARIOS      = {TRAIN_SCENARIOS}')
print(f'TRAIN_EPOCHS         = {TRAIN_EPOCHS}')
print(f'TRAIN_BATCH_SIZE     = {TRAIN_BATCH_SIZE}')
print()
print('Frenet env vars:')
for k in ['FRENET_SMART_CENTERLINE','FRENET_TANH_D','FRENET_TANH_D_SCALE']:
    print(f'  {k:<28} = {os.environ.get(k, "<not set>")!r}')
print()
print('Codebase: wimaan3/cs269-flow-planner (our modified fork)')
print('Run name:', f'{RUN_KINEMATIC}_best_seed{RUN_SEED}')
print('Expected: 5000 train scenarios from DagsHub, 50 epochs, batch 32')
print('Hypothesis: if best-ever 16.34 (1500 in-train) scales up to 5k val-log,')
print('  the Frenet penalty is data-limited; otherwise intrinsic-mismatch is confirmed.')
